# Multi-Agent RAG Email Reply System (Retriever → Writer → Reviewer, LangGraph)

This notebook extends a single-agent email-reply assistant into a **multi-agent system**
orchestrated with LangGraph:

- **Retriever** — searches an uploaded PDF (embedded in a Chroma vector store using Cohere
  embeddings) for content relevant to the incoming email.
- **Writer (Answerer)** — drafts a reply grounded in the retrieved document excerpts, using a
  local Gemma 3 model wrapped as a LangChain chat model.
- **Reviewer** — checks the draft for factual grounding and professionalism; if rejected, sends
  it back to the Writer with feedback (up to 3 retries before escalating to a human).

The reply is **never sent automatically** — a human-in-the-loop confirmation step lets the user
send, regenerate with feedback, edit manually, or quit without sending.

**Requirements:**
- A Hugging Face token with access to the gated `google/gemma-3-4b-it` model
- A Cohere API key (for embeddings)
- A Google Cloud OAuth `client_secret.json` file with the Gmail API enabled (scope: `gmail.modify`)
- A GPU runtime is strongly recommended for running the 4B-parameter model

⚠️ **Security note:** Never commit real API keys/tokens into this notebook. Use environment
variables, Colab secrets, or an untracked `.env` file instead of hardcoding them in cells below.


## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub pypdf \
    langchain langchain-core langchain-community langchain-text-splitters \
    langchain-cohere langchain-chroma langchain-huggingface langgraph cohere chromadb \
    google-api-python-client google-auth-httplib2 google-auth-oauthlib


## 2. Load the Language Model (Gemma 3 4B Instruct)

Model page: https://huggingface.co/google/gemma-3-4b-it

This model is **gated** — request access on the model page, then authenticate with a Hugging Face
token before it can be downloaded. The model is then wrapped as a LangChain-compatible chat model
so it can be used inside the LangGraph agent nodes below.


In [ ]:
from huggingface_hub import login

# Set your Hugging Face token as an environment variable / Colab secret instead of hardcoding
# it here. Example: login(token=os.environ["HF_TOKEN"])
login(token="")


In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, pipeline as hf_pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

MODEL_ID = "google/gemma-3-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, device_map="auto")

# Wrap the raw HF text-generation pipeline as a LangChain chat model, so it can be
# used with LangChain prompt templates and chains in the agent nodes below
text_gen_pipeline = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=processor.tokenizer,
    max_new_tokens=300,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
chat_model = ChatHuggingFace(llm=llm)
print("Gemma loaded and wrapped as a LangChain chat model.")


## 3. Embedding Model (Cohere)

Used to embed document chunks and queries into vectors for the retriever's similarity search.


In [ ]:
from langchain_cohere import CohereEmbeddings

# Set your Cohere API key as an environment variable / Colab secret instead of hardcoding
# it here. Example: CohereEmbeddings(cohere_api_key=os.environ["COHERE_API_KEY"], ...)
embeddings_model = CohereEmbeddings(cohere_api_key="", model="embed-english-v3.0")


## 4. Upload & Index the Reference Document

The uploaded PDF is the knowledge source the Retriever agent will search — e.g. a policy
document, product FAQ, or manual the email replies should be grounded in.


In [ ]:
from google.colab import files

print("Please upload your PDF document:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"\nUploaded: {filename}")


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the PDF and split it into overlapping chunks (overlap helps preserve context
# that would otherwise be cut across a chunk boundary)
loader = PyPDFLoader(filename)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"Loaded {len(documents)} pages, split into {len(chunks)} chunks")


In [ ]:
from langchain_chroma import Chroma

# Embed each chunk and store it in a local Chroma vector store; the retriever will
# return the top-k most similar chunks for a given query
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="my_document",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Document embedded and stored.")


## 5. Gmail Authentication

Runs an OAuth flow so the agent can read and send email on your behalf
(scope: `gmail.modify`).


In [ ]:
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
import pickle

SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]

# Path to the OAuth client secrets JSON file downloaded from Google Cloud Console
# (APIs & Services > Credentials > OAuth 2.0 Client ID > Desktop app)
CLIENT_SECRETS_FILE = ""

flow = Flow.from_client_secrets_file(
    CLIENT_SECRETS_FILE,
    scopes=SCOPES,
    redirect_uri="urn:ietf:wg:oauth:2.0:oob"
)

# Manual "out-of-band" OAuth flow: open the URL, sign in, approve access, and paste
# back the authorization code (used since there's no local redirect server in a notebook)
auth_url, _ = flow.authorization_url(prompt="consent")
print("Go to this URL, sign in, and approve access:\n")
print(auth_url)

code = input("\nPaste the authorization code here: ")
flow.fetch_token(code=code)
creds = flow.credentials

with open("token.pickle", "wb") as f:
    pickle.dump(creds, f)

gmail_service = build("gmail", "v1", credentials=creds)
print("\nGmail authenticated.")


## 6. Find & Fetch the Target Email

Lets the user search Gmail with a query, choose which matching email to reply to, then fetches
its full content (subject, sender, body, thread ID).


In [ ]:
def find_email(query):
    """
    Searches Gmail using the given query string, displays up to 5 matching messages
    (sender, subject, date), and prompts the user to choose which one to act on.
    Returns the Gmail message ID of the chosen email, or None if no matches were found.
    """
    results = gmail_service.users().messages().list(userId="me", q=query, maxResults=5).execute()
    messages = results.get("messages", [])
    if not messages:
        print("No matching email found.")
        return None

    print(f"Found {len(messages)} matches:\n")
    for i, m in enumerate(messages):
        msg = gmail_service.users().messages().get(
            userId="me", id=m["id"], format="metadata",
            metadataHeaders=["Subject", "From", "Date"]
        ).execute()
        headers = {h["name"]: h["value"] for h in msg["payload"]["headers"]}
        print(f"[{i}] From: {headers.get('From')} | Subject: {headers.get('Subject')} | Date: {headers.get('Date')}")

    choice = int(input("\nWhich one? (enter number): "))
    return messages[choice]["id"]


search_query = input("Search for email (e.g. 'from:someone@example.com subject:pricing'): ")
email_id = find_email(search_query)
print(f"\nSelected message ID: {email_id}")


In [ ]:
import base64

def get_email_content(msg_id):
    """
    Fetches the full content of a Gmail message by ID and extracts the fields the
    agent needs: subject, sender, plain-text body, thread ID (for replying in-thread),
    and the message ID itself.
    """
    msg = gmail_service.users().messages().get(userId="me", id=msg_id, format="full").execute()
    headers = msg["payload"]["headers"]
    subject = next((h["value"] for h in headers if h["name"] == "Subject"), "")
    sender = next((h["value"] for h in headers if h["name"] == "From"), "")
    thread_id = msg["threadId"]

    def get_body(payload):
        """Handles both single-part and multi-part Gmail message payloads, preferring
        the plain-text part when the message is multi-part."""
        if "parts" in payload:
            for part in payload["parts"]:
                if part["mimeType"] == "text/plain":
                    return base64.urlsafe_b64decode(part["body"]["data"]).decode("utf-8")
        elif "body" in payload and "data" in payload["body"]:
            return base64.urlsafe_b64decode(payload["body"]["data"]).decode("utf-8")
        return ""

    body = get_body(msg["payload"])
    return {"subject": subject, "sender": sender, "body": body, "thread_id": thread_id, "msg_id": msg_id}


email_data = get_email_content(email_id)
print(email_data)


## 7. Agent Tools

Two tools shared across the LangGraph agent nodes:
- `search_document` — the Retriever's interface into the Chroma vector store
- `send_email` — the only way the system can actually send mail; only ever invoked after
  explicit human approval in the confirmation loop at the end of the notebook


In [ ]:
from langchain_core.tools import tool
from email.mime.text import MIMEText


@tool
def search_document(question: str) -> str:
    """Search the uploaded document for content relevant to the question."""
    docs = retriever.invoke(question)
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


@tool
def send_email(to: str, subject: str, body: str, thread_id: str, msg_id: str) -> str:
    """Send an email reply via Gmail. Only call this after explicit human approval."""
    message = MIMEText(body)
    message["to"] = to
    message["subject"] = subject
    message["In-Reply-To"] = msg_id
    message["References"] = msg_id

    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()
    email_body = {"raw": raw, "threadId": thread_id}

    sent = gmail_service.users().messages().send(userId="me", body=email_body).execute()
    return f"Sent. Message ID: {sent['id']}"


## 8. Agent State

The shared state object that flows between every node in the LangGraph. Each node reads what
it needs and returns an updated copy of the state.


In [ ]:
from typing import TypedDict, List

class AgentState(TypedDict):
    email_sender: str
    email_subject: str
    email_body: str
    thread_id: str
    msg_id: str
    retrieved_chunks: List[str]
    answer: str
    review_feedback: str      # feedback from the Reviewer agent, if the draft was rejected
    human_feedback: str       # feedback from the human user, if they asked for a regeneration
    approved: bool
    retry_count: int
    escalated: bool           # True if the Reviewer never approved after max retries


## 9. Retriever Node

Searches the indexed document for content relevant to the incoming email's body, and stores the
resulting chunks in the shared state for the Writer to use as grounding context.


In [ ]:
def retriever_node(state: AgentState) -> AgentState:
    """Retrieves document chunks relevant to the email body and adds them to the state."""
    result = search_document.invoke({"question": state["email_body"]})
    chunks = result.split("\n\n---\n\n")

    print(f"[Retriever] Found {len(chunks)} chunks")
    return {**state, "retrieved_chunks": chunks}


## 10. Writer (Answerer) Node

Drafts a reply grounded in the retrieved document excerpts. If the Reviewer previously rejected
a draft, or the human requested changes, that feedback is folded into the prompt so the next
draft addresses it directly.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

answerer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional assistant that writes email replies. "
               "Use the provided document excerpts to ground your answer in accurate information. "
               "Answer ONLY what the sender specifically asked — do not include additional details, "
               "even if they're available in the excerpts, unless they were directly requested. "
               "If the document excerpts don't contain enough information to answer the sender's question, "
               "say so honestly in the reply instead of guessing or making up an answer. "
               "Start the reply with an appropriate greeting (e.g. 'Hi [Name],' or 'Hello,' if the sender's name isn't clear), "
               "and end with a closing such as 'Kind regards,' or 'Best regards,' followed by a blank line for the signature. "
               "Keep a professional but warm tone."),
    ("human", "Original email:\nFrom: {sender}\nSubject: {subject}\nBody: {body}\n\n"
              "Relevant document excerpts:\n{context}\n\n"
              "Write a professional reply, using the excerpts to answer accurately.{feedback_note}"),
])


def answerer_node(state: AgentState) -> AgentState:
    """Drafts (or redrafts) a reply, folding in Reviewer or human feedback if present."""
    context = "\n\n---\n\n".join(state["retrieved_chunks"])

    # Build a feedback note from whichever sources of feedback are present, so the
    # model explicitly knows what to fix in this attempt
    notes = []
    if state.get("review_feedback"):
        notes.append(f"Reviewer rejected the previous draft: {state['review_feedback']}")
    if state.get("human_feedback"):
        notes.append(f"Human requested this change: {state['human_feedback']}")
    feedback_note = ("\n\n" + "\n".join(notes) + "\nFix these in your new draft.") if notes else ""

    chain = answerer_prompt | chat_model
    response = chain.invoke({
        "sender": state["email_sender"], "subject": state["email_subject"],
        "body": state["email_body"], "context": context, "feedback_note": feedback_note,
    })

    print(f"[Answerer] Draft reply generated")
    return {**state, "answer": response.content.strip()}


## 11. Reviewer Node

Acts as a quality gate: checks the draft is factually grounded in the retrieved excerpts and
professionally written, before it's ever shown to the human for a send decision.


In [ ]:
reviewer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict reviewer checking an email reply. Verify: "
               "1) The reply is factually supported by the document excerpts. "
               "2) The reply is professional and directly addresses the original email. "
               "Respond with exactly 'APPROVED' if both pass, or 'REJECTED: <reason>' if either fails."),
    ("human", "Original email body: {body}\n\nDocument excerpts:\n{context}\n\nDraft reply to check: {answer}"),
])


def reviewer_node(state: AgentState) -> AgentState:
    """Reviews the current draft and records an approve/reject verdict (with reason) in the state."""
    context = "\n\n---\n\n".join(state["retrieved_chunks"])

    chain = reviewer_prompt | chat_model
    response = chain.invoke({"body": state["email_body"], "context": context, "answer": state["answer"]})
    verdict = response.content.strip()

    print(f"[Reviewer] Verdict: {verdict}")

    approved = verdict.upper().startswith("APPROVED")
    feedback = "" if approved else verdict.replace("REJECTED:", "").strip()

    return {**state, "approved": approved, "review_feedback": feedback, "retry_count": state.get("retry_count", 0) + 1}


## 12. Build the Graph

Wires the nodes together: Retriever → Writer → Reviewer, with a conditional loop back to the
Writer on rejection (up to 3 attempts), or escalation to a human-review flag if it's still not
approved after that.


In [ ]:
from langgraph.graph import StateGraph, END

def should_retry(state: AgentState) -> str:
    """Routing function: decides whether to end, retry the Writer, or escalate to a human,
    based on the Reviewer's verdict and how many attempts have already been made."""
    if state["approved"]:
        return "end"
    if state["retry_count"] >= 3:
        return "escalate"
    return "retry"


def escalate_node(state: AgentState) -> AgentState:
    """Marks the draft as escalated for human review after repeated Reviewer rejections."""
    print(f"[Escalation] Not approved after {state['retry_count']} attempts. Flagging for human review.")
    return {**state, "escalated": True}


graph = StateGraph(AgentState)
graph.add_node("retriever", retriever_node)
graph.add_node("answerer", answerer_node)
graph.add_node("reviewer", reviewer_node)
graph.add_node("escalate", escalate_node)

graph.set_entry_point("retriever")
graph.add_edge("retriever", "answerer")
graph.add_edge("answerer", "reviewer")
graph.add_conditional_edges("reviewer", should_retry, {"retry": "answerer", "escalate": "escalate", "end": END})
graph.add_edge("escalate", END)

app = graph.compile()
print("Graph compiled.")


## 13. Run the Agent Pipeline

Runs the full Retriever → Writer → Reviewer graph on the selected email and shows whether the
draft was approved or escalated.


In [ ]:
result = app.invoke({
    "email_sender": email_data["sender"], "email_subject": email_data["subject"],
    "email_body": email_data["body"], "thread_id": email_data["thread_id"], "msg_id": email_data["msg_id"],
    "retrieved_chunks": [], "answer": "", "review_feedback": "",
    "human_feedback": "",
    "approved": False, "retry_count": 0, "escalated": False,
})

print("\n--- Result ---")
print("ESCALATED" if result["escalated"] else f"Approved after {result['retry_count']} attempt(s)")
print(f"\nDraft:\n{result['answer']}")


## 14. Human-in-the-Loop Confirmation Loop

The final safeguard: regardless of the Reviewer's verdict, the email is never sent without
explicit human confirmation (with a second "are you sure" prompt on send). The user can also
regenerate the whole agent pipeline with feedback, edit the reply manually, or quit.


In [ ]:
reply_text = result["answer"]

while True:
    print("\n--- Reply ---")
    print(reply_text)
    if result["escalated"]:
        print("\n⚠️ ESCALATED — never approved by Reviewer. Review carefully.")
    print("-------------")

    choice = input("\n(s)end, (r)egenerate with feedback, (e)dit manually, (q)uit: ").strip().lower()

    if choice == "s":
        # Double confirmation before actually sending
        confirm = input("Confirm send? (y/n): ").strip().lower()
        if confirm == "y":
            outcome = send_email.invoke({
                "to": email_data["sender"], "subject": "Re: " + email_data["subject"],
                "body": reply_text, "thread_id": email_data["thread_id"], "msg_id": email_data["msg_id"],
            })
            print(outcome)
        break

    elif choice == "r":
        # Re-run the ENTIRE graph (retriever -> writer -> reviewer) with the human's
        # feedback included, rather than just re-prompting the writer in isolation
        human_feedback = input("What should change? (e.g. 'shorter', 'mention refund policy'): ")
        result = app.invoke({
            "email_sender": email_data["sender"], "email_subject": email_data["subject"],
            "email_body": email_data["body"], "thread_id": email_data["thread_id"], "msg_id": email_data["msg_id"],
            "retrieved_chunks": [], "answer": "", "review_feedback": "",
            "human_feedback": human_feedback,
            "approved": False, "retry_count": 0, "escalated": False,
        })
        reply_text = result["answer"]

    elif choice == "e":
        new_text = input("\nType your replacement reply: ")
        if new_text.strip():
            reply_text = new_text

    elif choice == "q":
        print("Cancelled. Nothing was sent.")
        break

    else:
        print("Invalid choice.")
